# MASI Full Dataset Pipeline

This notebook runs the proposal-aligned MASI pipeline for `configs/Full_dataset.json`.

It covers the complete prepared `full_dataset` Kaggle flow:

1. clone MASI from GitHub into `/kaggle/working/MASI`,
2. validate the prepared dataset at `/kaggle/input/datasets/dheerajrajanala/masi-amazon-csj-full-dataset`,
3. reuse the dataset's existing `images/` folder and manifests,
4. run Phase 1 behavior alignment, Phase 2 dual RQ-VAE tokenization, and Phase 3 recommendation training,
5. inspect summaries and package run outputs.

The attached dataset is expected to contain `Clothing_Shoes_and_Jewelry.jsonl`, `meta_Clothing_Shoes_and_Jewelry.jsonl`, `images/`, `image_download_manifest.json`, and `subset_manifest.json`. This notebook does not rebuild the prepared subset or redownload images on Kaggle.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


REPO_URL = "https://github.com/pradyunuydarp/MASI.git"
REPO_BRANCH = "main"
KAGGLE_WORKING_ROOT = Path("/kaggle/working")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
RUNNING_ON_KAGGLE = KAGGLE_WORKING_ROOT.exists() and KAGGLE_INPUT_ROOT.exists()
USE_GIT_CLONE_ON_KAGGLE = True
KAGGLE_REPO_DIR = KAGGLE_WORKING_ROOT / "MASI"
KAGGLE_FULL_DATASET_DIR = KAGGLE_INPUT_ROOT / "datasets" / "dheerajrajanala" / "masi-amazon-csj-full-dataset"
REQUIRED_DATASET_ENTRIES = [
    "Clothing_Shoes_and_Jewelry.jsonl",
    "meta_Clothing_Shoes_and_Jewelry.jsonl",
    "images",
    "image_download_manifest.json",
    "subset_manifest.json",
]


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "masi").exists():
            return candidate
    raise FileNotFoundError("Could not find the MASI repository root from this notebook location.")


if RUNNING_ON_KAGGLE and USE_GIT_CLONE_ON_KAGGLE:
    REPO_DIR = KAGGLE_REPO_DIR
    KAGGLE_WORKING_ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(KAGGLE_WORKING_ROOT)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
        cwd=KAGGLE_WORKING_ROOT,
    )
else:
    REPO_DIR = find_repo_root(Path.cwd())
os.chdir(REPO_DIR)

CONFIG_PATH = REPO_DIR / "configs" / "Full_dataset.json"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    FULL_DATASET_CONFIG = json.load(handle)

STORAGE_ROOT = KAGGLE_WORKING_ROOT / "masi_artifacts" if RUNNING_ON_KAGGLE else REPO_DIR
RAW_DIR = REPO_DIR / "data" / "raw" / "amazon_reviews_2023"
RAW_REVIEWS_PATH = RAW_DIR / "Clothing_Shoes_and_Jewelry.jsonl"
RAW_METADATA_PATH = RAW_DIR / "meta_Clothing_Shoes_and_Jewelry.jsonl"
PREPARED_DIR = REPO_DIR / "data" / "full_dataset"
RUN_ROOT = STORAGE_ROOT / "outputs" / "amazon_csj_full_dataset_train"

DATASET_CONFIG = FULL_DATASET_CONFIG["dataset"]
REVIEWS_RELPATH = DATASET_CONFIG.get("reviews_relpath") or "Clothing_Shoes_and_Jewelry.jsonl"
METADATA_RELPATH = DATASET_CONFIG.get("metadata_relpath") or "meta_Clothing_Shoes_and_Jewelry.jsonl"
ATTACHED_DATASET_DIR = None
if RUNNING_ON_KAGGLE:
    if all((KAGGLE_FULL_DATASET_DIR / entry).exists() for entry in REQUIRED_DATASET_ENTRIES):
        ATTACHED_DATASET_DIR = KAGGLE_FULL_DATASET_DIR
    for slug in ([] if ATTACHED_DATASET_DIR is not None else DATASET_CONFIG.get("kaggle_input_slugs", [])):
        candidate_paths = [KAGGLE_INPUT_ROOT / str(slug)]
        nested_root = KAGGLE_INPUT_ROOT / "datasets"
        if nested_root.exists():
            candidate_paths.extend(sorted(nested_root.glob(f"*/{slug}")))
        for candidate in candidate_paths:
            if (candidate / REVIEWS_RELPATH).exists() and (candidate / METADATA_RELPATH).exists():
                ATTACHED_DATASET_DIR = candidate
                break
        if ATTACHED_DATASET_DIR is not None:
            break

ACTIVE_PREPARED_DIR = ATTACHED_DATASET_DIR if ATTACHED_DATASET_DIR is not None else PREPARED_DIR
USING_ATTACHED_PREPARED_DATASET = ATTACHED_DATASET_DIR is not None

# Toggle these per runtime. The defaults avoid accidental large downloads and reruns.
RUN_PIP_INSTALL = True
PREFETCH_IMAGES = False
FORCE_PREPARE = False
FORCE_TRAIN = False
EXPORT_BUNDLE = True
IMAGE_WORKERS = 16

print(f"Repository: {REPO_DIR}")
print(f"Repo source: {REPO_URL if RUNNING_ON_KAGGLE and USE_GIT_CLONE_ON_KAGGLE else 'existing checkout'}")
print(f"Config:     {CONFIG_PATH}")
print(f"Storage:    {STORAGE_ROOT}")
print(f"Prepared:   {ACTIVE_PREPARED_DIR}")
print(f"Attached:   {ATTACHED_DATASET_DIR}")
print(f"Images:     {ACTIVE_PREPARED_DIR / 'images'}")
print(f"Run root:   {RUN_ROOT}")

In [ ]:
if RUN_PIP_INSTALL:
    pyproject_path = REPO_DIR / "pyproject.toml"
    pyproject_text = pyproject_path.read_text(encoding="utf-8")
    patched_pyproject_text = pyproject_text.replace('"numpy>=2.4.3"', '"numpy>=1.26,<2.1"')
    if patched_pyproject_text != pyproject_text:
        pyproject_path.write_text(patched_pyproject_text, encoding="utf-8")
        print("Patched pyproject.toml to keep NumPy compatible with Kaggle packages.")

    clip_source_path = REPO_DIR / "src" / "masi" / "tokenization" / "masi_tokens.py"
    clip_source_text = clip_source_path.read_text(encoding="utf-8")
    patched_clip_source_text = clip_source_text
    patched_clip_source_text = patched_clip_source_text.replace("import json\nfrom pathlib", "import json\nimport os\nfrom pathlib")
    patched_clip_source_text = patched_clip_source_text.replace(
        "    model = CLIPModel.from_pretrained(model_name).to(device)\n"
        "    processor = CLIPProcessor.from_pretrained(model_name)\n",
        "    hf_token = os.environ.get(\"HF_TOKEN\") or os.environ.get(\"HUGGING_FACE_HUB_TOKEN\") or None\n"
        "    model = CLIPModel.from_pretrained(\n"
        "        model_name,\n"
        "        token=hf_token,\n"
        "        low_cpu_mem_usage=False,\n"
        "        use_safetensors=True,\n"
        "    ).to(device)\n"
        "    processor = CLIPProcessor.from_pretrained(model_name, token=hf_token)\n",
    )
    if patched_clip_source_text != clip_source_text:
        clip_source_path.write_text(patched_clip_source_text, encoding="utf-8")
        print("Patched CLIP loader to avoid Kaggle meta-tensor materialization hangs.")

    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy>=1.26,<2.1"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[recommender]"], check=True)

    import numpy as np

    print(f"NumPy: {np.__version__}")
else:
    print("Skipping package installation.")

In [ ]:
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")

HF_CACHE_ROOT = STORAGE_ROOT / "hf_cache"
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
hf_env = {
    "HF_HOME": str(HF_CACHE_ROOT),
    "HF_HUB_CACHE": str(HF_CACHE_ROOT / "hub"),
    "TRANSFORMERS_CACHE": str(HF_CACHE_ROOT / "transformers"),
    "HF_HUB_DISABLE_XET": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
os.environ.update(hf_env)
env.update(hf_env)

if RUNNING_ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        env["HF_TOKEN"] = hf_token
        env["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Kaggle secrets.")
    except Exception as exc:
        print(f"HF_TOKEN not configured; continuing unauthenticated. {exc}")
else:
    print("Not running on Kaggle; using existing Hugging Face authentication, if any.")

import torch

if torch.cuda.is_available():
    device = f"cuda:{torch.cuda.current_device()}"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

usage = shutil.disk_usage(REPO_DIR)
print(f"Python: {sys.executable}")
print(f"Torch:  {torch.__version__}")
print(f"Device: {device}")
print(f"Free disk at repo: {usage.free / (1024 ** 3):.1f} GiB")
print(f"HF cache: {HF_CACHE_ROOT}")

assert CONFIG_PATH.exists(), f"Missing config: {CONFIG_PATH}"

In [ ]:
PRELOAD_CLIP_MODEL = True

if PRELOAD_CLIP_MODEL:
    import time
    from transformers import CLIPModel, CLIPProcessor

    clip_model_name = FULL_DATASET_CONFIG.get("clip", {}).get("model_name", "openai/clip-vit-base-patch32")
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or None
    started_at = time.time()
    print(f"Preloading CLIP assets from Hugging Face: {clip_model_name}")
    print(f"HF cache: {HF_CACHE_ROOT}")

    processor = CLIPProcessor.from_pretrained(clip_model_name, token=hf_token)
    model = CLIPModel.from_pretrained(
        clip_model_name,
        token=hf_token,
        low_cpu_mem_usage=False,
        use_safetensors=True,
    )
    del model, processor

    print(f"CLIP preload completed in {(time.time() - started_at) / 60:.1f} minutes.")
else:
    print("Skipping CLIP preload.")

## Prepared Kaggle Dataset

The Kaggle dataset is already prepared and includes reviews, metadata, images, and manifests. The notebook validates those files before training.

In [ ]:
required_dataset_paths = [ACTIVE_PREPARED_DIR / entry for entry in REQUIRED_DATASET_ENTRIES]
missing_dataset_paths = [path for path in required_dataset_paths if not path.exists()]

if missing_dataset_paths:
    raise FileNotFoundError(
        "Missing prepared dataset entries:\n"
        + "\n".join(f"- {path}" for path in missing_dataset_paths)
        + "\nExpected Kaggle dataset path: "
        + str(KAGGLE_FULL_DATASET_DIR)
    )
else:
    print("Prepared full_dataset input is complete.")
    for path in required_dataset_paths:
        print(f"- {path}")

## Prepare `data/full_dataset`

The `full_dataset` preset scans up to 20,000,000 review records, applies 5-core filtering, caps the selected subset at 102,400 users and 204,800 items, and writes a disk-backed SQLite selection index for lineage.

In [ ]:
prepared_reviews = ACTIVE_PREPARED_DIR / REVIEWS_RELPATH
prepared_metadata = ACTIVE_PREPARED_DIR / METADATA_RELPATH
subset_manifest = ACTIVE_PREPARED_DIR / "subset_manifest.json"

need_prepare = (not USING_ATTACHED_PREPARED_DATASET) and (
    FORCE_PREPARE or not (prepared_reviews.exists() and prepared_metadata.exists() and subset_manifest.exists())
)

if USING_ATTACHED_PREPARED_DATASET:
    print(f"Prepared full_dataset is attached read-only at {ACTIVE_PREPARED_DIR}.")
elif need_prepare:
    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable,
            "scripts/prepare_amazon_csj_subset.py",
            "--reviews-path",
            str(RAW_REVIEWS_PATH),
            "--metadata-path",
            str(RAW_METADATA_PATH),
            "--output-dir",
            str(PREPARED_DIR),
            "--preset",
            "full_dataset",
        ],
        check=True,
        env=env,
    )
else:
    print("Prepared full_dataset files already exist; set FORCE_PREPARE = True to rebuild.")

if subset_manifest.exists():
    with subset_manifest.open("r", encoding="utf-8") as handle:
        subset_summary = json.load(handle)
else:
    subset_summary = {"note": "subset_manifest.json was not found; training can still use the configured prepared JSONL files."}

print(json.dumps({
    "selected_user_count": subset_summary.get("selected_user_count"),
    "selected_item_count": subset_summary.get("selected_item_count"),
    "selected_interaction_count": subset_summary.get("selected_interaction_count"),
    "subset_manifest": str(subset_manifest),
}, indent=2))

## Download And Validate Images

This writes validated images to `data/full_dataset/images` and records image coverage in `image_download_manifest.json`.

In [ ]:
image_manifest = ACTIVE_PREPARED_DIR / "image_download_manifest.json"
configured_image_manifest = RUN_ROOT / "image_download_manifest.json"

if PREFETCH_IMAGES and USING_ATTACHED_PREPARED_DATASET:
    subprocess.run(
        [
            sys.executable,
            "scripts/download_amazon_csj_images.py",
            "--config",
            str(CONFIG_PATH),
            "--storage-root",
            str(STORAGE_ROOT),
            "--workers",
            str(IMAGE_WORKERS),
            "--retries",
            "2",
            "--timeout-seconds",
            "30",
            "--resume",
        ],
        check=True,
        env=env,
    )
elif PREFETCH_IMAGES:
    subprocess.run(
        [
            sys.executable,
            "scripts/download_amazon_csj_subset_images.py",
            "--metadata-path",
            str(prepared_metadata),
            "--output-dir",
            str(PREPARED_DIR),
            "--workers",
            str(IMAGE_WORKERS),
            "--retries",
            "2",
            "--timeout-seconds",
            "30",
            "--resume",
        ],
        check=True,
        env=env,
    )
else:
    print("Skipping image prefetch. train_masi.py can still download missing images if configured.")

summary_manifest = configured_image_manifest if configured_image_manifest.exists() else image_manifest
if summary_manifest.exists():
    with summary_manifest.open("r", encoding="utf-8") as handle:
        image_summary = json.load(handle)
    print(json.dumps({
        "selected_item_count": image_summary.get("selected_item_count"),
        "successful_item_count": image_summary.get("successful_item_count"),
        "failed_item_count": image_summary.get("failed_item_count"),
        "missing_url_item_count": image_summary.get("missing_url_item_count"),
        "image_manifest": str(summary_manifest),
    }, indent=2))

## Run MASI Training

This calls `scripts/train_masi.py --config configs/Full_dataset.json --storage-root .`. The launcher writes resolved configs, Phase 1/2 tokens, Phase 3 experiment artifacts, checkpoints, and a top-level run manifest.

In [ ]:
train_command = [
    sys.executable,
    "scripts/train_masi.py",
    "--config",
    str(CONFIG_PATH),
    "--storage-root",
    str(STORAGE_ROOT),
]
if FORCE_TRAIN:
    train_command.append("--force")

subprocess.run(train_command, check=True, env=env)

In [ ]:
run_manifest_path = RUN_ROOT / "run_manifest.json"
token_summary_path = RUN_ROOT / "phase12_tokens" / "masi_token_summary.json"
experiment_summary_path = RUN_ROOT / "phase3_experiment" / "experiment_summary.json"

for required_path in [run_manifest_path, token_summary_path, experiment_summary_path]:
    assert required_path.exists(), f"Expected artifact does not exist: {required_path}"

with run_manifest_path.open("r", encoding="utf-8") as handle:
    run_manifest = json.load(handle)
with token_summary_path.open("r", encoding="utf-8") as handle:
    token_summary = json.load(handle)
with experiment_summary_path.open("r", encoding="utf-8") as handle:
    experiment_summary = json.load(handle)

metrics = {
    "warm_metrics": experiment_summary.get("warm_metrics", {}),
    "cold_metrics": experiment_summary.get("cold_metrics", {}),
}
print(json.dumps({
    "run_manifest": str(run_manifest_path),
    "fused_ids_path": run_manifest.get("token_summary", {}).get("fused_ids_path"),
    "items_with_full_modalities": token_summary.get("items_with_full_modalities"),
    "metrics": metrics,
}, indent=2))

In [ ]:
checkpoint_root = RUN_ROOT / "checkpoints"
interesting_paths = [
    RUN_ROOT / "resolved_configs" / "token_build.json",
    RUN_ROOT / "resolved_configs" / "experiment.json",
    RUN_ROOT / "phase12_tokens" / "fused_semantic_ids.jsonl",
    RUN_ROOT / "phase12_tokens" / "behavior_alignment.pt",
    RUN_ROOT / "phase3_experiment" / "generative_recommender.pt",
    checkpoint_root,
]

for path in interesting_paths:
    status = "exists" if path.exists() else "missing"
    print(f"{status:7} {path}")

## Export Run Outputs

The export bundle includes the run outputs and checkpoints under `outputs/amazon_csj_full_dataset_train`. It does not include the raw files or `data/full_dataset/images`.

In [ ]:
if EXPORT_BUNDLE:
    archive_base = STORAGE_ROOT / "outputs" / "amazon_csj_full_dataset_train_artifacts"
    zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT)
    print(f"Exported run bundle: {zip_path}")
else:
    print("Skipping export bundle.")